# Selective KV Cache Loading - With Patches

This notebook clones LMCache and applies our modifications for selective loading.

In [ ]:
!nvidia-smi

In [ ]:
# Clone original LMCache
!git clone https://github.com/LMCache/LMCache.git
%cd LMCache

## Apply Patch 1: Content-Only Hashing

Change `_prefix_hash` to not chain hashes.

In [ ]:
# Read the file
with open('lmcache/v1/token_database.py', 'r') as f:
    content = f.read()

# Find and replace the _prefix_hash method
old_code = '''    def _prefix_hash(
        self,
        token_chunks: Iterable[Union[torch.Tensor, List[int]]],
    ) -> Iterable[int]:
        prefix_hash = self._get_init_hash()
        for token_chunk in token_chunks:
            prefix_hash = self._hash_tokens(token_chunk, prefix_hash)
            yield prefix_hash'''

new_code = '''    def _prefix_hash(
        self,
        token_chunks: Iterable[Union[torch.Tensor, List[int]]],
    ) -> Iterable[int]:
        # MODIFIED: Content-only hashing (no prefix chain)
        # This allows selective block loading without requiring all previous blocks
        for token_chunk in token_chunks:
            chunk_hash = self._hash_tokens(token_chunk)  # No prefix dependency!
            yield chunk_hash'''

if old_code in content:
    content = content.replace(old_code, new_code)
    with open('lmcache/v1/token_database.py', 'w') as f:
        f.write(content)
    print("✓ Patch 1 applied: Content-only hashing")
else:
    print("✗ Could not find exact match, trying alternative...")
    # Try simpler replacement
    if 'prefix_hash = self._hash_tokens(token_chunk, prefix_hash)' in content:
        content = content.replace(
            'prefix_hash = self._hash_tokens(token_chunk, prefix_hash)',
            'prefix_hash = self._hash_tokens(token_chunk)  # MODIFIED: content-only'
        )
        with open('lmcache/v1/token_database.py', 'w') as f:
            f.write(content)
        print("✓ Patch 1 applied (alternative): Content-only hashing")
    else:
        print("✗ Patch 1 failed")

## Apply Patch 2: Add hashes/offsets to process_tokens

In [ ]:
with open('lmcache/v1/token_database.py', 'r') as f:
    content = f.read()

# Find process_tokens and add hashes/offsets parameters
old_sig = '''    def process_tokens(
        self,
        tokens: Union[torch.Tensor, List[int]],'''

new_sig = '''    def process_tokens(
        self,
        tokens: Union[torch.Tensor, List[int]] = None,
        hashes: list = None,
        offsets: list = None,'''

if old_sig in content and 'hashes: list = None' not in content:
    content = content.replace(old_sig, new_sig)
    with open('lmcache/v1/token_database.py', 'w') as f:
        f.write(content)
    print("✓ Patch 2 applied: Added hashes/offsets parameters")
else:
    print("Patch 2: Already applied or signature different")

## Apply Patch 3: Hash-based processing in process_tokens

In [ ]:
with open('lmcache/v1/token_database.py', 'r') as f:
    content = f.read()

# Add hash-based early return at start of process_tokens body
hash_check = '''        # SELECTIVE LOADING: If hashes provided, yield directly
        if hashes is not None and offsets is not None:
            position = 0
            for h, offset in zip(hashes, offsets):
                yield (position, position + offset, h)
                position += offset
            return

'''

if 'SELECTIVE LOADING: If hashes provided' not in content:
    # Find where to insert - after make_key parameter in process_tokens
    marker = 'make_key: bool = True,\n    ) ->'
    if marker in content:
        # Find the end of the docstring after this
        idx = content.find(marker)
        # Find the closing """ of docstring
        docstring_end = content.find('"""', idx + len(marker))
        if docstring_end > 0:
            docstring_end = content.find('"""', docstring_end + 3) + 3
            # Insert after docstring
            content = content[:docstring_end] + '\n' + hash_check + content[docstring_end:]
            with open('lmcache/v1/token_database.py', 'w') as f:
                f.write(content)
            print("✓ Patch 3 applied: Hash-based processing")
        else:
            print("✗ Patch 3: Could not find docstring end")
    else:
        print("✗ Patch 3: Could not find marker")
else:
    print("Patch 3: Already applied")

## Apply Patch 4: Add helper to config.py

In [ ]:
config_addition = '''

def _validate_and_set_config_value(config, key, value):
    """Validate and set a config value dynamically."""
    if not hasattr(config, key):
        return False
    try:
        setattr(config, key, value)
        return True
    except Exception:
        return False
'''

with open('lmcache/v1/config.py', 'r') as f:
    content = f.read()

if '_validate_and_set_config_value' not in content:
    with open('lmcache/v1/config.py', 'a') as f:
        f.write(config_addition)
    print("✓ Patch 4 applied: Added _validate_and_set_config_value")
else:
    print("Patch 4: Already applied")

## Install Patched LMCache

In [ ]:
# Install WITHOUT CUDA extensions (set NO_CUDA_EXT=1)
!pip install torch transformers -q
!pip install aiofile aiofiles aiohttp msgspec pyyaml pyzmq redis safetensors sortedcontainers py-cpuinfo -q
!NO_CUDA_EXT=1 pip install -e . --no-deps --no-build-isolation -q
print("\n✓ LMCache installed (without CUDA extensions)!")

In [ ]:
!pip install vllm -q
print("\n✓ vLLM installed!")

## Test 1: Content-Only Hashing

In [ ]:
import torch
from lmcache.v1.token_database import ChunkedTokenDatabase

print("="*60)
print("TEST: Content-Only Hashing")
print("="*60)

db = ChunkedTokenDatabase()
db.chunk_size = 256
db.save_unfull_chunk = True

# Same content, different prefixes
target_chunk = torch.tensor([100] * 256)
prefix_a = torch.tensor([1] * 256)
prefix_b = torch.tensor([2] * 256)

tokens_a = torch.cat([prefix_a, target_chunk])
tokens_b = torch.cat([prefix_b, target_chunk])

results_a = list(db.process_tokens(tokens=tokens_a, make_key=False))
results_b = list(db.process_tokens(tokens=tokens_b, make_key=False))

# Second chunk should have same hash regardless of prefix
hash_a = results_a[1][2]
hash_b = results_b[1][2]

print(f"\nPrefix A = [1,1,1...], Prefix B = [2,2,2...]")
print(f"Target chunk = [100,100,100...]")
print(f"\nHash of target chunk after prefix A: {hash_a}")
print(f"Hash of target chunk after prefix B: {hash_b}")

if hash_a == hash_b:
    print("\n✓ PASS: Same content → Same hash (content-only hashing works!)")
else:
    print("\n✗ FAIL: Different hashes (still using prefix chain)")

## Test 2: Hash-Based Retrieval

In [ ]:
print("="*60)
print("TEST: Hash/UUID-Based Retrieval")
print("="*60)

db = ChunkedTokenDatabase()
db.chunk_size = 256
db.save_unfull_chunk = True

# Use UUIDs directly instead of tokens
uuid1 = hash("conversation-turn-1")
uuid2 = hash("conversation-turn-2")
uuid3 = hash("conversation-turn-3")

print(f"\nUUID 1: {uuid1}")
print(f"UUID 2: {uuid2}")
print(f"UUID 3: {uuid3}")

# Process by hashes, not tokens!
results = list(db.process_tokens(
    hashes=[uuid1, uuid2, uuid3],
    offsets=[256, 256, 256],
    make_key=False
))

print(f"\nResults: {len(results)} blocks")
for i, (start, end, h) in enumerate(results):
    print(f"  Block {i}: positions {start}-{end}, hash={h}")

if len(results) == 3 and results[0][2] == uuid1:
    print("\n✓ PASS: Hash-based retrieval works!")
else:
    print("\n✗ FAIL")

## Test 3: Selective Block Loading

In [ ]:
print("="*60)
print("TEST: Selective Block Loading (Skip Middle Blocks)")
print("="*60)

db = ChunkedTokenDatabase()
db.chunk_size = 256
db.save_unfull_chunk = True

# Only request blocks 0 and 2, SKIP block 1!
uuid0 = hash("system-prompt")
uuid2 = hash("user-preference")  # Skip whatever was in block 1

print(f"\nRequesting only:")
print(f"  Block 0 (system prompt): {uuid0}")
print(f"  Block 2 (user preference): {uuid2}")
print(f"  SKIPPING Block 1!")

results = list(db.process_tokens(
    hashes=[uuid0, uuid2],
    offsets=[256, 256],
    make_key=False
))

print(f"\nResults: {len(results)} blocks")
for i, (start, end, h) in enumerate(results):
    print(f"  Block {i}: positions {start}-{end}")

# With contiguous mapping, blocks should be at 0-256, 256-512
if results[0][0] == 0 and results[0][1] == 256 and results[1][0] == 256 and results[1][1] == 512:
    print("\n✓ PASS: Selective loading with contiguous mapping!")
    print("  Block 0 → positions 0-256")
    print("  Block 2 → positions 256-512 (contiguous, not 512-768!)")
else:
    print("\n✗ FAIL")

## Test 4: vLLM Inference

In [ ]:
from vllm import LLM, SamplingParams

print("Loading Qwen2-0.5B...")
llm = LLM(
    model="Qwen/Qwen2-0.5B",
    max_model_len=512,
    gpu_memory_utilization=0.5,
)
print("✓ Model loaded!")

In [ ]:
prompts = ["Hello, my name is", "The best programming language is"]
outputs = llm.generate(prompts, SamplingParams(max_tokens=30, temperature=0.7))

print("\n" + "="*60)
print("GENERATION RESULTS")
print("="*60)
for out in outputs:
    print(f"\nPrompt: {out.prompt}")
    print(f"Output: {out.outputs[0].text}")

## Summary

### What We Changed:

1. **Content-Only Hashing** (`token_database.py`)
   - Before: `hash(block2) = f(tokens2, hash(block1))` - chained
   - After: `hash(block2) = f(tokens2)` - independent

2. **Hash-Based Retrieval** (`token_database.py`)
   - Added `hashes` and `offsets` parameters to `process_tokens()`
   - Can now retrieve blocks by UUID instead of tokens

3. **Selective Loading**
   - Load only the blocks you need (e.g., blocks 0, 2 - skip block 1)
   - Contiguous mapping: selected blocks placed at positions 0, 1, 2...

### Use Case:
```
Conversation: [System] [Turn 1] [Turn 2] [Turn 3] [Turn 4]
                 0        1        2        3        4

Traditional: Must load 0,1,2,3,4 to use any
Selective:   Load just 0,3,4 (skip irrelevant turns 1,2)
```